In [144]:
import os
os.chdir("/Users/naginilandarhi/DSAICourse/Prof_Shankar_Venkatagiri/LinAlg4DataScience/faker_data_synthesizer")

In [145]:
import pandas as pd
import numpy as np

In [146]:
print(os.getcwd())

/Users/naginilandarhi/DSAICourse/Prof_Shankar_Venkatagiri/LinAlg4DataScience/faker_data_synthesizer


In [ ]:
r100000 = pd.read_csv('rows10000000/synthetic_data_5000000_rows_10_cols_13_clusters.csv')

In [148]:
r100000.head()

,ID,col_1,col_2,col_3,col_4,col_5,col_6,col_7,col_8,col_9,true_label
0,bdd640fb06674ad19c80317fa3b1799d,67.067261,666.883616,385.155998,320.734565,232.431912,591.527879,302.798124,-0.295559,30.922962,10
1,23b8c1e9392446debeb13b9046685257,813.190901,596.383561,744.982851,299.088524,961.614074,890.279719,757.888821,123.632349,423.180524,2
2,bd9c66b3ad3c4d6d9a3d1fa7bc8960a9,-57.756221,64.007096,644.363404,720.355655,965.710232,259.851623,305.992753,424.402402,111.766616,3
3,972a846916414f828b9d2434e465e150,14.306507,894.182086,392.085244,128.163524,251.720623,529.696281,99.847387,853.925065,728.226500,12
4,17fc695a07a04a6e8822e8f36c031199,4.469314,899.936225,400.930824,122.914523,233.738602,534.573639,101.076921,841.624223,721.946054,12


In [149]:
r100000.dtypes

ID             object
col_1         float64
col_2         float64
col_3         float64
col_4         float64
col_5         float64
col_6         float64
col_7         float64
col_8         float64
col_9         float64
true_label      int64
dtype: object

In [150]:
data_matrix_a = r100000.drop(columns=['ID']).to_numpy(dtype=np.float32)

In [151]:
data_matrix_a.shape

(10000000, 10)

In [152]:
ids = r100000["ID"].astype(str).to_numpy()

I need to do is to deal with id column. I will drop the id column from the data matrix. I will keep only the feature columns in the data matrix.

Now I need to identify which cluster each row belongs to. I do not want to use KMeans from sklearn. I want to implement KMeans from scratch. I will use the following steps:
1. Randomly select k rows from the data matrix as the initial centroids.
2. Assign each row to the nearest centroid based on Euclidean distance.
3. Recalculate the centroids as the mean of the rows assigned to each centroid.
4. Repeat steps 2 and 3 until convergence (i.e., the centroids do not change significantly between iterations). 

In [153]:
import time

In [154]:
def kmeans_cpu(X,
               K,
               initial_centroids=None,
               max_iters=100,
               tol=1e-4,
               seed=42):
    N, D = X.shape
    if initial_centroids is None:
        rng = np.random.default_rng(seed)
        indices = rng.choice(N, K, replace=False)
        centroids = X[indices].copy()

    else:
        centroids = initial_centroids.copy()
        centroids = centroids.astype(np.float32)

    assignments = np.full(N, -1, dtype=np.int32)   
    start_time = time.perf_counter()

    for iteration in range(max_iters):
        distances = np.linalg.norm(X[:, np.newaxis, :] - centroids[np.newaxis, :, :], axis=2)
        new_assignments = np.argmin(distances, axis=1)

        if np.array_equal(assignments, new_assignments):
            print(f"Converged after {iteration} iterations.")
            break

        assignments = new_assignments
        new_centroids = np.empty_like(centroids)

        for k in range(K):
            assigned_points = X[assignments == k]
            if len(assigned_points) > 0:
                new_centroids[k] = assigned_points.mean(axis=0)
                new_centroids[k] = new_centroids[k].astype(np.float32)

        if np.array_equal(centroids, new_centroids):
            print(f"Converged after {iteration} iterations.")
            break

        movement = np.linalg.norm(
            new_centroids - centroids,
            axis=1
        )

        max_movement = np.max(movement)
        centroids = new_centroids
        if max_movement < tol:
            break

    end_time = time.perf_counter()
    execution_time = end_time - start_time
    return {
        "assignments": assignments,
            "centroids": centroids,
            "execution_time": execution_time,
            "iterations": iteration + 1,
            "max_movement": max_movement
        }        


In [155]:
rng = np.random.default_rng(42)
initial_centroids = rng.choice(len(data_matrix_a), size=17, replace=False)
initial_centroids = data_matrix_a[initial_centroids]
print("Initial centroids:\n", initial_centroids)

Initial centroids:
 [[ 8.05268188e+02  5.90997131e+02  7.27891235e+02  2.93448822e+02
   9.73314148e+02  8.78091553e+02  7.56578125e+02  1.16773705e+02
   4.05479370e+02  2.00000000e+00]
 [ 8.19597229e+02  5.91107971e+02  7.24973816e+02  2.87911713e+02
   9.62578247e+02  8.64218018e+02  7.57042480e+02  1.09204498e+02
   4.17493011e+02  2.00000000e+00]
 [ 6.48436432e+01  6.64424316e+02  3.90512512e+02  3.23414124e+02
   2.30137253e+02  5.88006653e+02  2.95870911e+02 -2.81651974e+00
   2.57514038e+01  1.00000000e+01]
 [ 7.47601624e+02  3.78040314e+02  8.60142456e+02  6.81021729e+02
  -2.89047837e+00  9.67997620e+02  7.50586487e+02  7.63645386e+02
   3.63534050e+01  0.00000000e+00]
 [-5.22565689e+01  6.96690445e+01  6.55011963e+02  7.19449829e+02
   9.73955688e+02  2.65363007e+02  3.21265106e+02  4.12601288e+02
   1.06887886e+02  3.00000000e+00]
 [ 7.54571411e+02  3.84487061e+02  8.38227783e+02  6.67757629e+02
   9.90555096e+00  9.70455444e+02  7.24477905e+02  7.56865356e+02
   3.14735622

In [156]:
result = kmeans_cpu(data_matrix_a, K=17, initial_centroids=initial_centroids, max_iters=1000, tol=1e-4, seed=42)
print(f"Execution time: {result['execution_time']:.4f} seconds")
print(f"Iterations: {result['iterations']}")
print(f"Max movement: {result['max_movement']:.6f}")
print(f"Centroids:\n{result['centroids']}")

Execution time: 2102.9756 seconds
Iterations: 1000
Max movement: 2.783909
Centroids:
[[ 8.0310834e+02  5.9478387e+02  7.3402301e+02  2.9003137e+02
   9.6741400e+02  8.8226654e+02  7.5610022e+02  1.1423274e+02
   4.1339453e+02  2.0000000e+00]
 [ 8.1462585e+02  5.9484479e+02  7.3387158e+02  2.8989777e+02
   9.6106396e+02  8.8246118e+02  7.5611200e+02  1.1398014e+02
   4.1357849e+02  2.0000000e+00]
 [ 6.7591393e+01  6.6665015e+02  3.8882404e+02  3.1935284e+02
   2.3172629e+02  5.9256500e+02  2.9740564e+02 -3.5483310e+00
   2.9798878e+01  1.0000000e+01]
 [ 7.5059741e+02  3.8038956e+02  8.4815179e+02  6.7158667e+02
   6.7870933e-01  9.7426843e+02  7.3928009e+02  7.6734875e+02
   4.1900139e+01  0.0000000e+00]
 [ 1.7211206e+02  1.8863626e+02  7.7660889e+02  6.6030475e+02
   8.8539917e+02  3.2167740e+02  2.2971510e+02  4.6403577e+02
   3.9299465e+01  1.9997694e+00]
 [ 7.5229272e+02  3.8510495e+02  8.4017731e+02  6.7066718e+02
   6.5954142e+00  9.7222034e+02  7.3516882e+02  7.6198822e+02
   3.9

In [157]:
output = pd.DataFrame({
    "ID": ids,
    "Cluster": result["assignments"]
})

output.to_csv("kmeans_output.csv", index=False)

In [158]:
np.savetxt(
    "initial_centroids.csv",
    initial_centroids,
    delimiter=","
)

In [159]:
def representative_points(
    X,
    ids,
    centroids,
    assignments
):
    representatives = []

    for k in range(len(centroids)):

        indices = np.where(assignments == k)[0]

        cluster_points = X[indices]

        diff = cluster_points - centroids[k]

        distances = np.sum(
            diff * diff,
            axis=1
        )

        nearest_local_index = np.argmin(distances)

        original_index = indices[nearest_local_index]

        representatives.append({
            "cluster": k,
            "ID": ids[original_index],
            "distance_to_centroid":
                float(np.sqrt(distances[nearest_local_index]))
        })

    return representatives

In [160]:
representatives = representative_points(
    data_matrix_a,
    ids,
    result["centroids"],
    result["assignments"]
)

print(representatives)

[{'cluster': 0, 'ID': '8c368d09414a473ab7ea5b3e5a1688f9', 'distance_to_centroid': 3.952692747116089}, {'cluster': 1, 'ID': '5f09faf5de9a433481a8126546526649', 'distance_to_centroid': 3.5214593410491943}, {'cluster': 2, 'ID': '87637b7818dd49cf9ee766e435247ee7', 'distance_to_centroid': 1.0363919734954834}, {'cluster': 3, 'ID': '42dd9fe5a7d54d5db6b858efb240be0c', 'distance_to_centroid': 3.4304611682891846}, {'cluster': 4, 'ID': '99c80f745fb0433f9b75b7afe61bb16f', 'distance_to_centroid': 298.6172790527344}, {'cluster': 5, 'ID': '4c9c480b4e004e8a8ad7d9624108c3d1', 'distance_to_centroid': 3.6196770668029785}, {'cluster': 6, 'ID': 'a60a044528a74c0aa75459ccedd5f4fc', 'distance_to_centroid': 1.1639727354049683}, {'cluster': 7, 'ID': '8b2326afa24040b8be44f264ec00975b', 'distance_to_centroid': 4.518513202667236}, {'cluster': 8, 'ID': 'a2995c52bfa94664bdb1ccc7719eeef9', 'distance_to_centroid': 1.189327359199524}, {'cluster': 9, 'ID': 'bc15d87064ec4dfd8328fd0591960b29', 'distance_to_centroid': 1.69